# Automated Web Security Header Scanner
A beginner-friendly defensive cybersecurity project. It checks a website's HTTP response for selected security headers. Only scan websites you own or have permission to assess.


In [ ]:
!pip -q install requests


In [ ]:
"""
Automated Web Security Header Scanner
Educational project for checking basic HTTP security headers on websites you are authorized to assess.
"""
import requests
from urllib.parse import urlparse

SECURITY_HEADERS = {
    "Strict-Transport-Security": "Helps enforce HTTPS connections.",
    "Content-Security-Policy": "Helps reduce certain cross-site scripting and injection risks.",
    "X-Content-Type-Options": "Helps prevent MIME-type sniffing.",
    "X-Frame-Options": "Helps reduce clickjacking risks.",
    "Referrer-Policy": "Controls referrer information sent by browsers.",
    "Permissions-Policy": "Controls access to selected browser features.",
}

def normalize_url(url):
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    parsed = urlparse(url)
    if not parsed.netloc:
        raise ValueError("Please enter a valid website URL.")
    return url

def scan_website(url, timeout=10):
    url = normalize_url(url)
    try:
        response = requests.get(
            url,
            timeout=timeout,
            allow_redirects=True,
            headers={"User-Agent": "EducationalSecurityHeaderScanner/1.0"}
        )
    except requests.RequestException as exc:
        return {
            "url": url,
            "final_url": None,
            "status_code": None,
            "error": str(exc),
            "headers": {}
        }

    results = {}
    for header, purpose in SECURITY_HEADERS.items():
        results[header] = {
            "present": header in response.headers,
            "value": response.headers.get(header, ""),
            "purpose": purpose
        }

    return {
        "url": url,
        "final_url": response.url,
        "status_code": response.status_code,
        "error": None,
        "headers": results
    }

def print_report(report):
    print(f"Requested URL: {report['url']}")
    if report["error"]:
        print("Scan could not be completed:", report["error"])
        return
    print(f"Final URL: {report['final_url']}")
    print(f"HTTP status: {report['status_code']}")
    print("\nSecurity header results:")
    for header, details in report["headers"].items():
        status = "PRESENT" if details["present"] else "MISSING"
        print(f"- {header}: {status}")
        print(f"  Purpose: {details['purpose']}")
        if details["present"]:
            print(f"  Value: {details['value']}")

if __name__ == "__main__":
    target = input("Enter a website URL you are authorized to check (e.g., https://example.com): ")
    print_report(scan_website(target))


## Run the scanner
Run the code cell above, then run this cell and enter a website URL. A public test site such as `https://example.com` can be used for a basic demonstration. Results are informational; missing headers do not alone prove a website is vulnerable.


In [ ]:
target = input('Website URL (e.g., https://example.com): ')
result = scan_website(target)
print_report(result)
